In [8]:
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.validation import check_is_fitted

In [9]:
df = pd.read_csv('../data/raw/penguins_size.csv')
df['sex'].value_counts(dropna=False)

sex
MALE      168
FEMALE    165
NaN        10
.           1
Name: count, dtype: int64

In [10]:
TARGET = 'species'
X = df.drop(columns=['species', 'island'])   # island убираем: утечка, нашли в ПР-2
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # tuning knob: доля теста
    stratify=y,        # сохраняет доли видов в train и test (Chinstrap самый редкий)
    random_state=42,   # tuning knob: фиксирует разбиение, результат воспроизводим
)
print(X_train.shape, X_test.shape)

(275, 5) (69, 5)


In [11]:
class IQRClipper(BaseEstimator, TransformerMixin):
    """Обрезает выбросы до границ [Q1 - k*IQR, Q3 + k*IQR].
    fit только считает и запоминает границы, transform только применяет."""

    def __init__(self, factor=1.5):   # tuning knob: 1.5 классика, 3.0 режет только грубые выбросы
        self.factor = factor          # в __init__ только сохраняем параметры, ничего не считаем

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr   # подчёркивание в конце = «выучено в fit»
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        check_is_fitted(self, ['lower_', 'upper_'])   # ошибка, если забыли вызвать fit
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

In [12]:
num_cols = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
cat_cols = ['sex']

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),   # медиана устойчива к выбросам
    ('clip',   IQRClipper(factor=1.5)),             # наш трансформер
    ('scale',  StandardScaler()),                   # после обрезки, чтобы выбросы не искажали среднее
])
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='unknown')),  # пропуск = отдельная категория, не 0
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocess = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])

In [13]:
X_train_prep = preprocess.fit_transform(X_train)   # fit ТОЛЬКО на train
X_test_prep  = preprocess.transform(X_test)        # на test только transform, без fit
print(X_train_prep.shape, X_test_prep.shape)
assert X_train_prep.shape[1] == X_test_prep.shape[1]
assert not np.isnan(X_train_prep).any() and not np.isnan(X_test_prep).any()

(275, 8) (69, 8)


In [ ]:
def test_bounds_frozen_after_fit():
    clipper = IQRClipper().fit(X_train[num_cols])
    low, up = clipper.lower_.copy(), clipper.upper_.copy()
    out = clipper.transform(np.vstack([up + 1000, low - 1000]))   # заведомые выбросы
    assert np.array_equal(clipper.lower_, low) and np.array_equal(clipper.upper_, up)  # transform не меняет границы
    assert np.allclose(out[0], up) and np.allclose(out[1], low)        # обрезано ровно до границ train

def test_values_inside_bounds_unchanged():
    clipper = IQRClipper().fit(X_train[num_cols])
    inside = ((clipper.lower_ + clipper.upper_) / 2).reshape(1, -1)
    assert np.array_equal(clipper.transform(inside), inside)           # внутри границ ничего не трогаем

test_bounds_frozen_after_fit()
test_values_inside_bounds_unchanged()
print('Оба теста пройдены')

AssertionError: 

In [ ]:
def make_big(X, n, noise=0.05, seed=42):   # tuning knob: noise, доля std для шума
    """n строк из X с возвращением + небольшой шум на числовых столбцах."""
    rng = np.random.default_rng(seed)
    big = X.sample(n=n, replace=True, random_state=seed).reset_index(drop=True)
    for c in num_cols:
        big[c] = big[c] + rng.normal(0, noise * X[c].std(), size=n)
    return big

rows = []
for n in [1_000, 10_000, 100_000, 1_000_000]:   # tuning knob: размеры
    big = make_big(X_train, n)
    t0 = time.perf_counter()
    clone(preprocess).fit_transform(big)         # clone: обучаем копию, боевой пайплайн не перетираем
    rows.append({'n_rows': n, 'seconds': round(time.perf_counter() - t0, 3)})
pd.DataFrame(rows)

In [ ]:
def timeit(label, fn):
    t0 = time.perf_counter(); out = fn()
    print(f'{label:<16}{time.perf_counter() - t0:.3f} c'); return out

big = make_big(X_train, 1_000_000)
a = timeit('median impute', lambda: SimpleImputer(strategy='median').fit_transform(big[num_cols]))
b = timeit('IQR clip',      lambda: IQRClipper().fit_transform(a))
_ = timeit('scaler',        lambda: StandardScaler().fit_transform(b))
c = timeit('cat impute',    lambda: SimpleImputer(strategy='constant', fill_value='unknown').fit_transform(big[cat_cols]))
_ = timeit('one-hot',       lambda: OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit_transform(c))